In [ ]:
# ============================================================
# BASELINE CASCADE EVALUATION
# ============================================================

import json
import os
import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
from transformers import (
    AutoModel,
    AutoModelForQuestionAnswering,
    AutoModelForTokenClassification,
    AutoTokenizer,
)

# ============================================================
# 1. LOCAL COLAB CONFIGURATION & PATHS
# ============================================================

MODEL_NAME = "dicta-il/dictabert"

VAL_FILE = "/content/val_set.csv"
STAGE3_PATH = "/content/stage3_model"
LABEL_ENCODER_FILE = os.path.join(STAGE3_PATH, "label_encoders.json")

ATTRIBUTE_COLS = ["dim", "tr", "fr", "sr", "fe", "refLevel", "ss"]

STAGE1_MAX_LEN = 128
STAGE2_MAX_LEN = 160
MAX_REF_SPAN = 12

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


# ============================================================
# 2. NORMALIZATION
# ============================================================


def normalize_hebrew(text):
    if text is None:
        return ""
    text = str(text)
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ============================================================
# 3. LOAD VALIDATION DATA
# ============================================================

print("\n" + "=" * 70)
print("LOADING VALIDATION DATA")
print("=" * 70)

if not os.path.exists(VAL_FILE):
    raise FileNotFoundError(
        f"Validation file not found at '{VAL_FILE}'. Please upload 'val_set.csv' to /content/."
    )

val_df = pd.read_csv(VAL_FILE)

print("Validation rows:", len(val_df))
print("Columns:", list(val_df.columns))

required_cols = [
    "Prompt_Word",
    "Full_Raw_Sentence",
    "MeaningVal",
    "Referent",
] + ATTRIBUTE_COLS

missing = [c for c in required_cols if c not in val_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

for col in ["Prompt_Word", "Full_Raw_Sentence", "MeaningVal", "Referent"]:
    val_df[col] = val_df[col].fillna("").astype(str)


# ============================================================
# 4. LOAD TOKENIZER
# ============================================================

print("\nLoading DictaBERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded.")


# ============================================================
# 5. BASELINE STAGE 1
# ============================================================

print("\n" + "=" * 70)
print("CREATING BASELINE STAGE 1")
print("=" * 70)

baseline_stage1 = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=3
)
baseline_stage1.to(DEVICE)
baseline_stage1.eval()

for param in baseline_stage1.parameters():
    param.requires_grad = False

ID_TO_BIO = {0: "O", 1: "B-MU", 2: "I-MU"}

print("Stage 1 created (Frozen pretrained backbone + Random Head).")


# ============================================================
# 6. BASELINE STAGE 2
# ============================================================

print("\n" + "=" * 70)
print("CREATING BASELINE STAGE 2")
print("=" * 70)

baseline_stage2 = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)
baseline_stage2.to(DEVICE)
baseline_stage2.eval()

for param in baseline_stage2.parameters():
    param.requires_grad = False

print("Stage 2 created (Frozen pretrained backbone + Random QA Heads).")


# ============================================================
# 7. BASELINE STAGE 3
# ============================================================


class BaselineStage3(nn.Module):

    def __init__(self, num_labels_dict):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.heads = nn.ModuleDict()

        for col in ATTRIBUTE_COLS:
            self.heads[col] = nn.Linear(hidden_size, num_labels_dict[col])

    def mean_pooling(self, hidden_states, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        masked_hidden = hidden_states * mask
        summed = masked_hidden.sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.mean_pooling(outputs.last_hidden_state, attention_mask)
        pooled = self.dropout(pooled)
        logits = {}
        for col in ATTRIBUTE_COLS:
            logits[col] = self.heads[col](pooled)
        return logits


# ============================================================
# 8. LOAD LABEL ENCODERS
# ============================================================

print("\nLoading Stage 3 label encoders...")
label_encoders = {}

if os.path.exists(LABEL_ENCODER_FILE):
    with open(LABEL_ENCODER_FILE, "r", encoding="utf-8") as f:
        encoder_data = json.load(f)

    for col in ATTRIBUTE_COLS:
        if col not in encoder_data:
            raise ValueError(
                f"Column '{col}' missing from label_encoders.json"
            )
        classes = encoder_data[col]
        le = LabelEncoder()
        le.classes_ = np.array(classes, dtype=object)
        label_encoders[col] = le
    print("Loaded label encoders from:", LABEL_ENCODER_FILE)
else:
    print(
        "WARNING: label_encoders.json not found at location. Building encoders dynamically from validation set..."
    )
    for col in ATTRIBUTE_COLS:
        values = val_df[col].dropna().astype(str).unique()
        le = LabelEncoder()
        le.fit(values)
        label_encoders[col] = le

num_labels_dict = {}
for col in ATTRIBUTE_COLS:
    num_labels_dict[col] = len(label_encoders[col].classes_)
    print(f"  • {col:10s}: {num_labels_dict[col]} classes")


# ============================================================
# 9. INSTANTIATE BASELINE STAGE 3
# ============================================================

baseline_stage3 = BaselineStage3(num_labels_dict)
baseline_stage3.to(DEVICE)
baseline_stage3.eval()

for param in baseline_stage3.parameters():
    param.requires_grad = False

print("\nStage 3 created successfully.")


# ============================================================
# 10. INFERENCE HELPER FUNCTIONS
# ============================================================


@torch.no_grad()
def predict_stage1(sentence):
    encoding = tokenizer(
        sentence,
        max_length=STAGE1_MAX_LEN,
        truncation=True,
        padding="max_length",
        return_offsets_mapping=True,
        return_tensors="pt",
    )
    offsets = encoding.pop("offset_mapping")[0].cpu().numpy()
    model_inputs = {k: v.to(DEVICE) for k, v in encoding.items()}

    outputs = baseline_stage1(**model_inputs)
    predictions = torch.argmax(outputs.logits, dim=-1)[0].cpu().numpy()

    spans = []
    current_start = None
    current_end = None

    for token_idx, label_id in enumerate(predictions):
        label = ID_TO_BIO.get(int(label_id), "O")
        start, end = offsets[token_idx]

        if end <= start:
            continue

        if label == "B-MU":
            if current_start is not None:
                span_text = normalize_hebrew(
                    sentence[current_start:current_end]
                )
                if span_text:
                    spans.append(span_text)
            current_start = int(start)
            current_end = int(end)

        elif label == "I-MU":
            if current_start is not None:
                current_end = int(end)

        else:
            if current_start is not None:
                span_text = normalize_hebrew(
                    sentence[current_start:current_end]
                )
                if span_text:
                    spans.append(span_text)
                current_start = None
                current_end = None

    if current_start is not None:
        span_text = normalize_hebrew(sentence[current_start:current_end])
        if span_text:
            spans.append(span_text)

    unique_spans = []
    for span in spans:
        if span not in unique_spans:
            unique_spans.append(span)

    return unique_spans


@torch.no_grad()
def predict_stage2(sentence, meaning_value):
    question = f"מהו הרפרנט של יחידת המשמעות: {meaning_value} ?"
    context = f"גירוי: {meaning_value.split()[0] if meaning_value else ''} | משפט: {sentence}"

    encoding = tokenizer(
        question,
        context,
        max_length=STAGE2_MAX_LEN,
        truncation="only_second",
        padding="max_length",
        return_token_type_ids=True,
        return_tensors="pt",
    )
    model_inputs = {k: v.to(DEVICE) for k, v in encoding.items()}

    outputs = baseline_stage2(**model_inputs)

    start_logits = outputs.start_logits[0]
    end_logits = outputs.end_logits[0]
    input_ids = encoding["input_ids"][0]
    attention_mask = encoding["attention_mask"][0]
    token_type_ids = encoding.get("token_type_ids", [None])[0]

    valid_positions = []
    for i in range(len(input_ids)):
        if attention_mask[i].item() == 0:
            continue
        if token_type_ids is not None and token_type_ids[i].item() != 1:
            continue
        valid_positions.append(i)

    if not valid_positions:
        return ""

    best_score = -float("inf")
    best_start = None
    best_end = None

    for start in valid_positions:
        max_end = min(start + MAX_REF_SPAN - 1, valid_positions[-1])
        for end in valid_positions:
            if end < start:
                continue
            if end > max_end:
                break
            score = start_logits[start].item() + end_logits[end].item()
            if score > best_score:
                best_score = score
                best_start = start
                best_end = end

    if best_start is None:
        return ""

    selected_ids = input_ids[best_start : best_end + 1]
    text = tokenizer.decode(selected_ids, skip_special_tokens=True)
    return normalize_hebrew(text)


@torch.no_grad()
def predict_stage3(sentence, meaning_value, referent):
    text = f"משפט: {sentence} | יחידה: {meaning_value} | רפרנט: {referent}"

    encoding = tokenizer(
        text,
        max_length=128,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )
    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    logits = baseline_stage3(
        input_ids=input_ids, attention_mask=attention_mask
    )
    predictions = {}

    for col in ATTRIBUTE_COLS:
        pred_id = torch.argmax(logits[col], dim=-1).item()
        predictions[col] = label_encoders[col].inverse_transform([pred_id])[0]

    return predictions


# ============================================================
# 11. PREPARE UNIQUE SENTENCES
# ============================================================

unique_sentences = val_df["Full_Raw_Sentence"].drop_duplicates().tolist()
print("\nUnique validation sentences:", len(unique_sentences))

# Cache Stage 1 Predictions
print("\nRunning Baseline Stage 1...")
stage1_predictions = {s: predict_stage1(s) for s in unique_sentences}

# Cache Stage 2 Predictions for End-to-End
print("Running Baseline Stage 2 (for E2E)...")
stage2_predictions = {}
total_stage2 = 0
for sentence in unique_sentences:
    stage2_predictions[sentence] = {}
    for mv in stage1_predictions[sentence]:
        stage2_predictions[sentence][mv] = predict_stage2(sentence, mv)
        total_stage2 += 1


# ============================================================
# 18. STAGE 1 BASELINE EVALUATION
# ============================================================

print("\n" + "=" * 70)
print("BASELINE STAGE 1 EVALUATION")
print("=" * 70)

gold_mvs_by_sentence = {}
for sentence, group in val_df.groupby("Full_Raw_Sentence"):
    gold_mvs = set()
    for mv in group["MeaningVal"]:
        mv = normalize_hebrew(mv)
        if mv:
            gold_mvs.add(mv)
    gold_mvs_by_sentence[sentence] = gold_mvs

sentence_exact_matches = 0
sentence_total = len(unique_sentences)
total_correct_mvs = 0
total_predicted_mvs = 0
total_gold_mvs = 0

for sentence in unique_sentences:
    predicted_mvs = set(
        normalize_hebrew(mv)
        for mv in stage1_predictions.get(sentence, [])
        if normalize_hebrew(mv)
    )

    gold_mvs = gold_mvs_by_sentence.get(sentence, set())

    if predicted_mvs == gold_mvs:
        sentence_exact_matches += 1

    correct = len(predicted_mvs.intersection(gold_mvs))
    total_correct_mvs += correct
    total_predicted_mvs += len(predicted_mvs)
    total_gold_mvs += len(gold_mvs)

stage1_sentence_exact = (
    100.0 * sentence_exact_matches / sentence_total
    if sentence_total > 0
    else 0.0
)
stage1_mv_precision = (
    100.0 * total_correct_mvs / total_predicted_mvs
    if total_predicted_mvs > 0
    else 0.0
)
stage1_mv_recall = (
    100.0 * total_correct_mvs / total_gold_mvs if total_gold_mvs > 0 else 0.0
)

print("\n" + "-" * 70)
print("BASELINE STAGE 1 RESULTS")
print("-" * 70)
print(
    f"  • Sentence Exact-Set Match : {stage1_sentence_exact:.2f}% ({sentence_exact_matches}/{sentence_total})"
)
print(
    f"  • MV Precision             : {stage1_mv_precision:.2f}% ({total_correct_mvs}/{total_predicted_mvs})"
)
print(
    f"  • MV Recall                : {stage1_mv_recall:.2f}% ({total_correct_mvs}/{total_gold_mvs})"
)


# ============================================================
# 19. STAGE 2 BASELINE EVALUATION
# ============================================================

print("\n" + "=" * 70)
print("BASELINE STAGE 2 EVALUATION")
print("=" * 70)

stage2_gold_mv_predictions = {}
stage2_exact_correct = 0
stage2_exact_total = 0

gold_sentence_mv_pairs = val_df[
    ["Full_Raw_Sentence", "MeaningVal"]
].drop_duplicates()
print("Unique sentence + MV pairs:", len(gold_sentence_mv_pairs))

for idx, row in gold_sentence_mv_pairs.iterrows():
    sentence = row["Full_Raw_Sentence"]
    gold_mv = normalize_hebrew(row["MeaningVal"])

    if not gold_mv:
        continue

    predicted_ref = predict_stage2(sentence, gold_mv)
    predicted_ref = normalize_hebrew(predicted_ref)

    matching_rows = val_df[
        (val_df["Full_Raw_Sentence"] == sentence)
        & (val_df["MeaningVal"].apply(normalize_hebrew) == gold_mv)
    ]

    if len(matching_rows) == 0:
        continue

    gold_ref = normalize_hebrew(matching_rows.iloc[0]["Referent"])

    if not gold_ref:
        continue

    stage2_exact_total += 1

    if predicted_ref == gold_ref:
        stage2_exact_correct += 1

    stage2_gold_mv_predictions[(sentence, gold_mv)] = predicted_ref

stage2_exact_match = (
    100.0 * stage2_exact_correct / stage2_exact_total
    if stage2_exact_total > 0
    else 0.0
)

print("\n" + "-" * 70)
print("BASELINE STAGE 2 RESULTS")
print("-" * 70)
print(
    f"  • Referent Exact Match : {stage2_exact_match:.2f}% ({stage2_exact_correct}/{stage2_exact_total})"
)


# ============================================================
# STAGE 3 EVALUATION (ORACLE & END-TO-END)
# ============================================================

# --- ORACLE EVALUATION ---
oracle_correct = {col: 0 for col in ATTRIBUTE_COLS}
oracle_total = {col: 0 for col in ATTRIBUTE_COLS}

for _, row in val_df.iterrows():
    sentence = row["Full_Raw_Sentence"]
    gold_mv = row["MeaningVal"]
    gold_ref = row["Referent"]

    if not gold_mv or not gold_ref:
        continue

    predictions = predict_stage3(sentence, gold_mv, gold_ref)

    for col in ATTRIBUTE_COLS:
        gold_value = row[col]
        if pd.isna(gold_value):
            continue
        oracle_total[col] += 1
        if predictions[col] == str(gold_value):
            oracle_correct[col] += 1

# --- END-TO-END EVALUATION ---
e2e_correct = {col: 0 for col in ATTRIBUTE_COLS}
e2e_total = {col: 0 for col in ATTRIBUTE_COLS}
matched_mv_count = 0
total_gold_mv_rows = 0
matched_ref_count = 0

for _, row in val_df.iterrows():
    sentence = row["Full_Raw_Sentence"]
    gold_mv = normalize_hebrew(row["MeaningVal"])
    gold_ref = normalize_hebrew(row["Referent"])

    if not gold_mv:
        continue

    total_gold_mv_rows += 1
    predicted_mvs = stage1_predictions.get(sentence, [])
    predicted_mvs_normalized = [normalize_hebrew(mv) for mv in predicted_mvs]

    if gold_mv not in predicted_mvs_normalized:
        continue

    matched_mv_count += 1
    predicted_ref = normalize_hebrew(
        stage2_predictions.get(sentence, {}).get(gold_mv, "")
    )

    if predicted_ref == "":
        continue

    matched_ref_count += 1
    predictions = predict_stage3(sentence, gold_mv, predicted_ref)

    for col in ATTRIBUTE_COLS:
        gold_value = row[col]
        if pd.isna(gold_value):
            continue
        e2e_total[col] += 1
        if predictions[col] == str(gold_value):
            e2e_correct[col] += 1

oracle_results = {
    col: (
        (100.0 * oracle_correct[col] / oracle_total[col])
        if oracle_total[col] > 0
        else 0.0
    )
    for col in ATTRIBUTE_COLS
}
e2e_results = {
    col: (
        (100.0 * e2e_correct[col] / e2e_total[col])
        if e2e_total[col] > 0
        else 0.0
    )
    for col in ATTRIBUTE_COLS
}

oracle_macro = np.mean(list(oracle_results.values()))
e2e_macro = (
    np.mean(list(e2e_results.values())) if any(e2e_total.values()) else 0.0
)
mv_match_rate = (
    (100.0 * matched_mv_count / total_gold_mv_rows)
    if total_gold_mv_rows > 0
    else 0.0
)


# ============================================================
# 20. FINAL BASELINE RESULTS
# ============================================================

print("\n" + "=" * 70)
print("BASELINE CASCADE FINAL RESULTS")
print("=" * 70)

print("\nSTAGE 1 — MEANING VALUE EXTRACTION")
print("-" * 70)
print(f"  • Sentence Exact-Set Match : {stage1_sentence_exact:.2f}%")
print(f"  • MV Precision             : {stage1_mv_precision:.2f}%")
print(f"  • MV Recall                : {stage1_mv_recall:.2f}%")

print("\nSTAGE 2 — REFERENT EXTRACTION")
print("-" * 70)
print(f"  • Referent Exact Match     : {stage2_exact_match:.2f}%")

print("\nSTAGE 3 — ATTRIBUTE CLASSIFICATION")
print("-" * 70)
for col in ATTRIBUTE_COLS:
    print(
        f"  • {col:10s} Oracle: {oracle_results[col]:6.2f}%   E2E: {e2e_results[col]:6.2f}%"
    )

print("\n" + "-" * 70)
print(f"  • Stage 3 Oracle Macro: {oracle_macro:.2f}%")
print(f"  • Stage 3 E2E Macro:    {e2e_macro:.2f}%")
print(f"  • Stage 1 MV Match Rate: {mv_match_rate:.2f}%")
print("=" * 70)


# ============================================================
# 23. SAVE RESULTS
# ============================================================

RESULTS_FILE = "/content/baseline_cascade_stage3_results.json"
results_payload = {
    "stage1": {
        "exact_set_match": stage1_sentence_exact,
        "precision": stage1_mv_precision,
        "recall": stage1_mv_recall,
    },
    "stage2": {"exact_match": stage2_exact_match},
    "stage3": {
        "oracle": oracle_results,
        "e2e": e2e_results,
        "oracle_macro": oracle_macro,
        "e2e_macro": e2e_macro,
        "mv_match_rate": mv_match_rate,
    },
}

with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(results_payload, f, ensure_ascii=False, indent=2)

print(f"\nResults saved locally to: {RESULTS_FILE}")

Using device: cuda

LOADING VALIDATION DATA
Validation rows: 4569
Columns: ['Original_ID', 'Prompt_Word', 'Full_Raw_Sentence', 'MeaningVal', 'Referent', 'dim', 'tr', 'fr', 'sr', 'fe', 'refLevel', 'ss']

Loading DictaBERT tokenizer...
Tokenizer loaded.

CREATING BASELINE STAGE 1


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dicta-il/dictabert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Stage 1 created (Frozen pretrained backbone + Random Head).

CREATING BASELINE STAGE 2


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: dicta-il/dictabert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Stage 2 created (Frozen pretrained backbone + Random QA Heads).

Loading Stage 3 label encoders...
  • dim       : 29 classes
  • tr        : 13 classes
  • fr        : 13 classes
  • sr        : 12 classes
  • fe        : 3 classes
  • refLevel  : 6 classes
  • ss        : 10 classes


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dicta-il/dictabert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Stage 3 created successfully.

Unique validation sentences: 803

Running Baseline Stage 1...
Running Baseline Stage 2 (for E2E)...

BASELINE STAGE 1 EVALUATION

----------------------------------------------------------------------
BASELINE STAGE 1 RESULTS
----------------------------------------------------------------------
  • Sentence Exact-Set Match : 3.24% (26/803)
  • MV Precision             : 31.34% (320/1021)
  • MV Recall                : 7.16% (320/4468)

BASELINE STAGE 2 EVALUATION
Unique sentence + MV pairs: 4477

----------------------------------------------------------------------
BASELINE STAGE 2 RESULTS
----------------------------------------------------------------------
  • Referent Exact Match : 0.36% (16/4477)

BASELINE CASCADE FINAL RESULTS

STAGE 1 — MEANING VALUE EXTRACTION
----------------------------------------------------------------------
  • Sentence Exact-Set Match : 3.24%
  • MV Precision             : 31.34%
  • MV Recall                : 7.16%

STA

On Test

In [ ]:
# ============================================================
# BASELINE CASCADE EVALUATION ON TEST SET
# ============================================================

import json
import os
import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
from transformers import (
    AutoModel,
    AutoModelForQuestionAnswering,
    AutoModelForTokenClassification,
    AutoTokenizer,
)

# ============================================================
# 1. LOCAL COLAB CONFIGURATION & PATHS
# ============================================================

MODEL_NAME = "dicta-il/dictabert"

TEST_FILE = (
    "/content/test_set.csv"
    if os.path.exists("/content/test_set.csv")
    else "stage3_splits/test_set.csv"
)

STAGE3_PATH = "/content/stage3_model"
LABEL_ENCODER_FILE = os.path.join(STAGE3_PATH, "label_encoders.json")

ATTRIBUTE_COLS = ["dim", "tr", "fr", "sr", "fe", "refLevel", "ss"]

STAGE1_MAX_LEN = 128
STAGE2_MAX_LEN = 160
MAX_REF_SPAN = 12

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


# ============================================================
# 2. NORMALIZATION
# ============================================================


def normalize_hebrew(text):
    if text is None:
        return ""
    text = str(text)
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ============================================================
# 3. LOAD TEST DATA
# ============================================================

print("\n" + "=" * 70)
print("LOADING TEST DATA")
print("=" * 70)

if not os.path.exists(TEST_FILE):
    raise FileNotFoundError(
        f"Test file not found at '{TEST_FILE}'. Please upload 'test_set.csv' to /content/ or 'stage3_splits/test_set.csv'."
    )

test_df = pd.read_csv(TEST_FILE)

print("Test rows:", len(test_df))
print("Columns:", list(test_df.columns))

required_cols = [
    "Prompt_Word",
    "Full_Raw_Sentence",
    "MeaningVal",
    "Referent",
] + ATTRIBUTE_COLS

missing = [c for c in required_cols if c not in test_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

for col in ["Prompt_Word", "Full_Raw_Sentence", "MeaningVal", "Referent"]:
    test_df[col] = test_df[col].fillna("").astype(str)


# ============================================================
# 4. LOAD TOKENIZER
# ============================================================

print("\nLoading DictaBERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Tokenizer loaded.")


# ============================================================
# 5. BASELINE STAGE 1
# ============================================================

print("\n" + "=" * 70)
print("CREATING BASELINE STAGE 1")
print("=" * 70)

baseline_stage1 = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=3
)
baseline_stage1.to(DEVICE)
baseline_stage1.eval()

for param in baseline_stage1.parameters():
    param.requires_grad = False

ID_TO_BIO = {0: "O", 1: "B-MU", 2: "I-MU"}

print("Stage 1 created (Frozen pretrained backbone + Random Head).")


# ============================================================
# 6. BASELINE STAGE 2
# ============================================================

print("\n" + "=" * 70)
print("CREATING BASELINE STAGE 2")
print("=" * 70)

baseline_stage2 = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)
baseline_stage2.to(DEVICE)
baseline_stage2.eval()

for param in baseline_stage2.parameters():
    param.requires_grad = False

print("Stage 2 created (Frozen pretrained backbone + Random QA Heads).")


# ============================================================
# 7. BASELINE STAGE 3 ARCHITECTURE
# ============================================================


class BaselineStage3(nn.Module):

    def __init__(self, num_labels_dict):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)
        hidden_size = self.bert.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.heads = nn.ModuleDict()

        for col in ATTRIBUTE_COLS:
            self.heads[col] = nn.Linear(hidden_size, num_labels_dict[col])

    def mean_pooling(self, hidden_states, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        masked_hidden = hidden_states * mask
        summed = masked_hidden.sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.mean_pooling(outputs.last_hidden_state, attention_mask)
        pooled = self.dropout(pooled)
        logits = {}
        for col in ATTRIBUTE_COLS:
            logits[col] = self.heads[col](pooled)
        return logits


# ============================================================
# 8. LOAD LABEL ENCODERS
# ============================================================

print("\nLoading Stage 3 label encoders...")
label_encoders = {}

if os.path.exists(LABEL_ENCODER_FILE):
    with open(LABEL_ENCODER_FILE, "r", encoding="utf-8") as f:
        encoder_data = json.load(f)

    for col in ATTRIBUTE_COLS:
        if col not in encoder_data:
            raise ValueError(
                f"Column '{col}' missing from label_encoders.json"
            )
        classes = encoder_data[col]
        le = LabelEncoder()
        le.classes_ = np.array(classes, dtype=object)
        label_encoders[col] = le
    print("Loaded label encoders from:", LABEL_ENCODER_FILE)
else:
    print(
        "WARNING: label_encoders.json not found at location. Building encoders dynamically from test set..."
    )
    for col in ATTRIBUTE_COLS:
        values = test_df[col].dropna().astype(str).unique()
        le = LabelEncoder()
        le.fit(values)
        label_encoders[col] = le

num_labels_dict = {}
for col in ATTRIBUTE_COLS:
    num_labels_dict[col] = len(label_encoders[col].classes_)
    print(f"  • {col:10s}: {num_labels_dict[col]} classes")


# ============================================================
# 9. INSTANTIATE BASELINE STAGE 3
# ============================================================

baseline_stage3 = BaselineStage3(num_labels_dict)
baseline_stage3.to(DEVICE)
baseline_stage3.eval()

for param in baseline_stage3.parameters():
    param.requires_grad = False

print("\nStage 3 created successfully.")


# ============================================================
# 10. INFERENCE HELPER FUNCTIONS
# ============================================================


@torch.no_grad()
def predict_stage1(sentence):
    encoding = tokenizer(
        sentence,
        max_length=STAGE1_MAX_LEN,
        truncation=True,
        padding="max_length",
        return_offsets_mapping=True,
        return_tensors="pt",
    )
    offsets = encoding.pop("offset_mapping")[0].cpu().numpy()
    model_inputs = {k: v.to(DEVICE) for k, v in encoding.items()}

    outputs = baseline_stage1(**model_inputs)
    predictions = torch.argmax(outputs.logits, dim=-1)[0].cpu().numpy()

    spans = []
    current_start = None
    current_end = None

    for token_idx, label_id in enumerate(predictions):
        label = ID_TO_BIO.get(int(label_id), "O")
        start, end = offsets[token_idx]

        if end <= start:
            continue

        if label == "B-MU":
            if current_start is not None:
                span_text = normalize_hebrew(
                    sentence[current_start:current_end]
                )
                if span_text:
                    spans.append(span_text)
            current_start = int(start)
            current_end = int(end)

        elif label == "I-MU":
            if current_start is not None:
                current_end = int(end)

        else:
            if current_start is not None:
                span_text = normalize_hebrew(
                    sentence[current_start:current_end]
                )
                if span_text:
                    spans.append(span_text)
                current_start = None
                current_end = None

    if current_start is not None:
        span_text = normalize_hebrew(sentence[current_start:current_end])
        if span_text:
            spans.append(span_text)

    unique_spans = []
    for span in spans:
        if span not in unique_spans:
            unique_spans.append(span)

    return unique_spans


@torch.no_grad()
def predict_stage2(sentence, meaning_value):
    question = f"מהו הרפרנט של יחידת המשמעות: {meaning_value} ?"
    context = f"גירוי: {meaning_value.split()[0] if meaning_value else ''} | משפט: {sentence}"

    encoding = tokenizer(
        question,
        context,
        max_length=STAGE2_MAX_LEN,
        truncation="only_second",
        padding="max_length",
        return_token_type_ids=True,
        return_tensors="pt",
    )
    model_inputs = {k: v.to(DEVICE) for k, v in encoding.items()}

    outputs = baseline_stage2(**model_inputs)

    start_logits = outputs.start_logits[0]
    end_logits = outputs.end_logits[0]
    input_ids = encoding["input_ids"][0]
    attention_mask = encoding["attention_mask"][0]
    token_type_ids = encoding.get("token_type_ids", [None])[0]

    valid_positions = []
    for i in range(len(input_ids)):
        if attention_mask[i].item() == 0:
            continue
        if token_type_ids is not None and token_type_ids[i].item() != 1:
            continue
        valid_positions.append(i)

    if not valid_positions:
        return ""

    best_score = -float("inf")
    best_start = None
    best_end = None

    for start in valid_positions:
        max_end = min(start + MAX_REF_SPAN - 1, valid_positions[-1])
        for end in valid_positions:
            if end < start:
                continue
            if end > max_end:
                break
            score = start_logits[start].item() + end_logits[end].item()
            if score > best_score:
                best_score = score
                best_start = start
                best_end = end

    if best_start is None:
        return ""

    selected_ids = input_ids[best_start : best_end + 1]
    text = tokenizer.decode(selected_ids, skip_special_tokens=True)
    return normalize_hebrew(text)


@torch.no_grad()
def predict_stage3(sentence, meaning_value, referent):
    text = f"משפט: {sentence} | יחידה: {meaning_value} | רפרנט: {referent}"

    encoding = tokenizer(
        text,
        max_length=128,
        truncation=True,
        padding="max_length",
        return_tensors="pt",
    )
    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    logits = baseline_stage3(
        input_ids=input_ids, attention_mask=attention_mask
    )
    predictions = {}

    for col in ATTRIBUTE_COLS:
        pred_id = torch.argmax(logits[col], dim=-1).item()
        predictions[col] = label_encoders[col].inverse_transform([pred_id])[0]

    return predictions


# ============================================================
# 11. PREPARE UNIQUE SENTENCES
# ============================================================

unique_sentences = test_df["Full_Raw_Sentence"].drop_duplicates().tolist()
print("\nUnique test sentences:", len(unique_sentences))

# Cache Stage 1 Predictions
print("\nRunning Baseline Stage 1...")
stage1_predictions = {s: predict_stage1(s) for s in unique_sentences}

# Cache Stage 2 Predictions for End-to-End
print("Running Baseline Stage 2 (for E2E)...")
stage2_predictions = {}
total_stage2 = 0
for sentence in unique_sentences:
    stage2_predictions[sentence] = {}
    for mv in stage1_predictions[sentence]:
        stage2_predictions[sentence][mv] = predict_stage2(sentence, mv)
        total_stage2 += 1


# ============================================================
# 12. STAGE 1 BASELINE EVALUATION
# ============================================================

print("\n" + "=" * 70)
print("BASELINE STAGE 1 EVALUATION (TEST SET)")
print("=" * 70)

gold_mvs_by_sentence = {}
for sentence, group in test_df.groupby("Full_Raw_Sentence"):
    gold_mvs = set()
    for mv in group["MeaningVal"]:
        mv = normalize_hebrew(mv)
        if mv:
            gold_mvs.add(mv)
    gold_mvs_by_sentence[sentence] = gold_mvs

sentence_exact_matches = 0
sentence_total = len(unique_sentences)
total_correct_mvs = 0
total_predicted_mvs = 0
total_gold_mvs = 0

for sentence in unique_sentences:
    predicted_mvs = set(
        normalize_hebrew(mv)
        for mv in stage1_predictions.get(sentence, [])
        if normalize_hebrew(mv)
    )

    gold_mvs = gold_mvs_by_sentence.get(sentence, set())

    if predicted_mvs == gold_mvs:
        sentence_exact_matches += 1

    correct = len(predicted_mvs.intersection(gold_mvs))
    total_correct_mvs += correct
    total_predicted_mvs += len(predicted_mvs)
    total_gold_mvs += len(gold_mvs)

stage1_sentence_exact = (
    100.0 * sentence_exact_matches / sentence_total
    if sentence_total > 0
    else 0.0
)
stage1_mv_precision = (
    100.0 * total_correct_mvs / total_predicted_mvs
    if total_predicted_mvs > 0
    else 0.0
)
stage1_mv_recall = (
    100.0 * total_correct_mvs / total_gold_mvs if total_gold_mvs > 0 else 0.0
)

print("\n" + "-" * 70)
print("BASELINE STAGE 1 RESULTS")
print("-" * 70)
print(
    f"  • Sentence Exact-Set Match : {stage1_sentence_exact:.2f}% ({sentence_exact_matches}/{sentence_total})"
)
print(
    f"  • MV Precision             : {stage1_mv_precision:.2f}% ({total_correct_mvs}/{total_predicted_mvs})"
)
print(
    f"  • MV Recall                : {stage1_mv_recall:.2f}% ({total_correct_mvs}/{total_gold_mvs})"
)


# ============================================================
# 13. STAGE 2 BASELINE EVALUATION
# ============================================================

print("\n" + "=" * 70)
print("BASELINE STAGE 2 EVALUATION (TEST SET)")
print("=" * 70)

stage2_gold_mv_predictions = {}
stage2_exact_correct = 0
stage2_exact_total = 0

gold_sentence_mv_pairs = test_df[
    ["Full_Raw_Sentence", "MeaningVal"]
].drop_duplicates()
print("Unique sentence + MV pairs:", len(gold_sentence_mv_pairs))

for idx, row in gold_sentence_mv_pairs.iterrows():
    sentence = row["Full_Raw_Sentence"]
    gold_mv = normalize_hebrew(row["MeaningVal"])

    if not gold_mv:
        continue

    predicted_ref = predict_stage2(sentence, gold_mv)
    predicted_ref = normalize_hebrew(predicted_ref)

    matching_rows = test_df[
        (test_df["Full_Raw_Sentence"] == sentence)
        & (test_df["MeaningVal"].apply(normalize_hebrew) == gold_mv)
    ]

    if len(matching_rows) == 0:
        continue

    gold_ref = normalize_hebrew(matching_rows.iloc[0]["Referent"])

    if not gold_ref:
        continue

    stage2_exact_total += 1

    if predicted_ref == gold_ref:
        stage2_exact_correct += 1

    stage2_gold_mv_predictions[(sentence, gold_mv)] = predicted_ref

stage2_exact_match = (
    100.0 * stage2_exact_correct / stage2_exact_total
    if stage2_exact_total > 0
    else 0.0
)

print("\n" + "-" * 70)
print("BASELINE STAGE 2 RESULTS")
print("-" * 70)
print(
    f"  • Referent Exact Match : {stage2_exact_match:.2f}% ({stage2_exact_correct}/{stage2_exact_total})"
)


# ============================================================
# 14. STAGE 3 EVALUATION (ORACLE & END-TO-END)
# ============================================================

print("\n" + "=" * 70)
print("BASELINE STAGE 3 EVALUATION (TEST SET)")
print("=" * 70)

# --- ORACLE EVALUATION ---
oracle_correct = {col: 0 for col in ATTRIBUTE_COLS}
oracle_total = {col: 0 for col in ATTRIBUTE_COLS}

for _, row in test_df.iterrows():
    sentence = row["Full_Raw_Sentence"]
    gold_mv = row["MeaningVal"]
    gold_ref = row["Referent"]

    if not gold_mv or not gold_ref:
        continue

    predictions = predict_stage3(sentence, gold_mv, gold_ref)

    for col in ATTRIBUTE_COLS:
        gold_value = row[col]
        if pd.isna(gold_value):
            continue
        oracle_total[col] += 1
        if predictions[col] == str(gold_value):
            oracle_correct[col] += 1

# --- END-TO-END EVALUATION ---
e2e_correct = {col: 0 for col in ATTRIBUTE_COLS}
e2e_total = {col: 0 for col in ATTRIBUTE_COLS}
matched_mv_count = 0
total_gold_mv_rows = 0
matched_ref_count = 0

for _, row in test_df.iterrows():
    sentence = row["Full_Raw_Sentence"]
    gold_mv = normalize_hebrew(row["MeaningVal"])
    gold_ref = normalize_hebrew(row["Referent"])

    if not gold_mv:
        continue

    total_gold_mv_rows += 1
    predicted_mvs = stage1_predictions.get(sentence, [])
    predicted_mvs_normalized = [normalize_hebrew(mv) for mv in predicted_mvs]

    if gold_mv not in predicted_mvs_normalized:
        continue

    matched_mv_count += 1
    predicted_ref = normalize_hebrew(
        stage2_predictions.get(sentence, {}).get(gold_mv, "")
    )

    if predicted_ref == "":
        continue

    matched_ref_count += 1
    predictions = predict_stage3(sentence, gold_mv, predicted_ref)

    for col in ATTRIBUTE_COLS:
        gold_value = row[col]
        if pd.isna(gold_value):
            continue
        e2e_total[col] += 1
        if predictions[col] == str(gold_value):
            e2e_correct[col] += 1

oracle_results = {
    col: (
        (100.0 * oracle_correct[col] / oracle_total[col])
        if oracle_total[col] > 0
        else 0.0
    )
    for col in ATTRIBUTE_COLS
}
e2e_results = {
    col: (
        (100.0 * e2e_correct[col] / e2e_total[col])
        if e2e_total[col] > 0
        else 0.0
    )
    for col in ATTRIBUTE_COLS
}

oracle_macro = np.mean(list(oracle_results.values()))
e2e_macro = (
    np.mean(list(e2e_results.values())) if any(e2e_total.values()) else 0.0
)
mv_match_rate = (
    (100.0 * matched_mv_count / total_gold_mv_rows)
    if total_gold_mv_rows > 0
    else 0.0
)


# ============================================================
# 15. FINAL BASELINE RESULTS
# ============================================================

print("\n" + "=" * 70)
print("BASELINE CASCADE FINAL TEST RESULTS")
print("=" * 70)

print("\nSTAGE 1 — MEANING VALUE EXTRACTION")
print("-" * 70)
print(f"  • Sentence Exact-Set Match : {stage1_sentence_exact:.2f}%")
print(f"  • MV Precision             : {stage1_mv_precision:.2f}%")
print(f"  • MV Recall                : {stage1_mv_recall:.2f}%")

print("\nSTAGE 2 — REFERENT EXTRACTION")
print("-" * 70)
print(f"  • Referent Exact Match     : {stage2_exact_match:.2f}%")

print("\nSTAGE 3 — ATTRIBUTE CLASSIFICATION")
print("-" * 70)
for col in ATTRIBUTE_COLS:
    print(
        f"  • {col:10s} Oracle: {oracle_results[col]:6.2f}%   E2E: {e2e_results[col]:6.2f}%"
    )

print("\n" + "-" * 70)
print(f"  • Stage 3 Oracle Macro: {oracle_macro:.2f}%")
print(f"  • Stage 3 E2E Macro:    {e2e_macro:.2f}%")
print(f"  • Stage 1 MV Match Rate: {mv_match_rate:.2f}%")
print("=" * 70)


# ============================================================
# 16. SAVE RESULTS
# ============================================================

RESULTS_FILE = "/content/baseline_cascade_stage3_test_results.json"
results_payload = {
    "stage1": {
        "exact_set_match": stage1_sentence_exact,
        "precision": stage1_mv_precision,
        "recall": stage1_mv_recall,
    },
    "stage2": {"exact_match": stage2_exact_match},
    "stage3": {
        "oracle": oracle_results,
        "e2e": e2e_results,
        "oracle_macro": oracle_macro,
        "e2e_macro": e2e_macro,
        "mv_match_rate": mv_match_rate,
    },
}

with open(RESULTS_FILE, "w", encoding="utf-8") as f:
    json.dump(results_payload, f, ensure_ascii=False, indent=2)

print(f"\nResults saved locally to: {RESULTS_FILE}")

Using device: cuda

LOADING TEST DATA
Test rows: 4114
Columns: ['Original_ID', 'Prompt_Word', 'Full_Raw_Sentence', 'MeaningVal', 'Referent', 'dim', 'tr', 'fr', 'sr', 'fe', 'refLevel', 'ss']

Loading DictaBERT tokenizer...


config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: reconstructing file:   0%|          |  0.00B / 1.50MB            

vocab.txt: downloading bytes:           |  0.00B            

...

[transformers] BertForQuestionAnswering LOAD REPORT from: dicta-il/dictabert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Stage 2 created (Frozen pretrained backbone + Random QA Heads).

Loading Stage 3 label encoders...
  • dim       : 30 classes
  • tr        : 12 classes
  • fr        : 16 classes
  • sr        : 10 classes
  • fe        : 1 classes
  • refLevel  : 6 classes
  • ss        : 8 classes


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dicta-il/dictabert
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Stage 3 created successfully.

Unique test sentences: 796

Running Baseline Stage 1...
Running Baseline Stage 2 (for E2E)...

BASELINE STAGE 1 EVALUATION (TEST SET)

----------------------------------------------------------------------
BASELINE STAGE 1 RESULTS
----------------------------------------------------------------------
  • Sentence Exact-Set Match : 15.95% (127/796)
  • MV Precision             : 30.18% (1890/6263)
  • MV Recall                : 47.18% (1890/4006)

BASELINE STAGE 2 EVALUATION (TEST SET)
Unique sentence + MV pairs: 4015

----------------------------------------------------------------------
BASELINE STAGE 2 RESULTS
----------------------------------------------------------------------
  • Referent Exact Match : 0.22% (9/4015)

BASELINE STAGE 3 EVALUATION (TEST SET)

BASELINE CASCADE FINAL TEST RESULTS

STAGE 1 — MEANING VALUE EXTRACTION
----------------------------------------------------------------------
  • Sentence Exact-Set Match : 15.95%
  • MV Precis